# 2.1 手写 JSON Schema：properties 与 required 的正确关系

**学完你能**：为任意工具手写合规 schema；一眼看出错在哪；理解「schema 即契约」。

Sources: agent-expert-outline.md 思维 #4 · lecture-notes.md Slide 3–4 · 参考 json-schema.org/learn/getting-started-step-by-step

## 概念导读：schema 是给模型看的参数说明书

function calling 协议里，每个工具附一段 JSON Schema 声明「我接受什么参数」。模型读它构造调用。

**最易错点**：`required` 是 schema 顶层的字段，与 `properties` 平级——不是塞在某个参数定义里。

```json
✗ {"properties": {"expr": {"type": "string", "required": true}}}
✓ {"type": "object", "properties": {"expr": {"type": "string"}}, "required": ["expr"]}
```

In [ ]:
# 手写：给 calc 工具配 schema（对照 agent/tools/base.py 的 json_schema()）
import json

calc_schema = {
    "type": "object",
    "properties": {
        "expr": {"type": "string", "description": "算术表达式，如 3*(7+2)。只支持四则、括号、**。"},
    },
    "required": ["expr"],
}
print(json.dumps(calc_schema, ensure_ascii=False, indent=2))

In [ ]:
# 校验你的 schema：手写一个 10 行的迷你校验器（不装 jsonschema 库——看透本质）

def check_schema(s: dict) -> list[str]:
    """返回错误列表，空列表 = 合格。只查本课关心的三点。"""
    errs = []
    if s.get("type") != "object":
        errs.append("顶层 type 必须是 object")
    if "properties" not in s:
        errs.append("缺少 properties")
    for name, spec in s.get("properties", {}).items():
        if "required" in spec:
            errs.append(f"字段 {name} 里混进了 required——它属于顶层")
    for name in s.get("required", []):
        if name not in s.get("properties", {}):
            errs.append(f"required 里的 {name} 在 properties 中不存在")
    return errs

print("正确示例:", check_schema(calc_schema))
print("错误示例:", check_schema({"properties": {"expr": {"type": "string", "required": True}}}))

In [ ]:
# 与 Tool 类对接：Tool.json_schema() 的输出应与你手写的一致
from agent.builtin.calc import calc
from agent.tools.base import Tool

t = Tool(name="calc", description="安全求值算术表达式", run=calc,
         parameters=calc_schema["properties"], required=("expr",))
print(t.json_schema() == calc_schema)   # True = 手写与实现达成契约
print(t.prompt_line())                  # 提示行：模型实际读到的一行

## 练习（答案只进 solutions 子仓）

1. **改参数**：给 calc_schema 加枚举限制（如限定 description 里声明的运算符集合），观察 check_schema 反应——枚举是 poka-yoke 的第一工具。
2. **破坏性实验**：把 required 改成 `["不存在"]`，看 check_schema 报什么——这是真实模型会遇到的服务端 400 错误的微缩版。
3. **扩展题**：为「查询天气」工具手写 schema：city(string, required) + date(string, optional, 格式 YYYY-MM-DD)，并用 check_schema 验证。

## 提交前检查

```bash
uv run pytest -q && uvx ruff check . && uv run python smoke.py
jupyter nbconvert --clear-output --inplace ch02-tools/2.1-schema.ipynb
```